# Allelic Dosage Effect Analysis (GWAS → Dose–Response)

This notebook implements an end-to-end pipeline to evaluate whether GWAS-identified genetic signals exhibit a dose–response relationship with phenotypic traits.

# Objective

To test whether increasing allelic dosage (0–4, tetraploid-capped) is associated with systematic changes in a phenotype across varieties, and to quantify the direction, strength, and shape of this relationship.

In [0]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql import functions as F, Window
import yaml
import sys

In [0]:



module_path = "/Volumes/bmqg/default_bronze/fatemeh/final_project/modules"

if module_path not in sys.path:
    sys.path.insert(0, module_path)


if "effect_gemma" in sys.modules:
    del sys.modules["effect_gemma"]

import effect_size_gemma
importlib.reload(effect_size_gemma)

print("Loaded from:", effect_size_gemma.__file__)


Loaded from: /Volumes/bmqg/default_bronze/fatemeh/final_project/modules/effect_size_gemma.py


In [0]:


# =========================
# Load config
# =========================
CONFIG_PATH = "/Volumes/bmqg/default_bronze/fatemeh/config_mixed.yaml"

with open(CONFIG_PATH, "r") as f:
    CONFIG = yaml.safe_load(f)

PHENO_PATH = CONFIG["paths"]["aroma_matrix_newharvested"]

assert os.path.exists(PHENO_PATH), "Phenotype file not found"

print("Phenotype file:", PHENO_PATH)


# =========================
# Read phenotype
# =========================
df_pheno = pd.read_csv(PHENO_PATH)

print("Raw phenotype shape:", df_pheno.shape)


# =========================
# Clean Variety column
# =========================
df_pheno["Variety"] = (
    df_pheno["Variety"]
    .astype(str)
    .str.strip()
    .str.upper()
)


# =========================
# Convert traits to numeric
# =========================
for c in df_pheno.columns:
    if c != "Variety":
        df_pheno[c] = pd.to_numeric(df_pheno[c], errors="coerce")


# =========================
# Check duplicates
# =========================
print("\nChecking replicate varieties...")

print("Total rows:", len(df_pheno))
print("Unique varieties:", df_pheno["Variety"].nunique())

dup = df_pheno["Variety"].value_counts()
print("\nReplicates:")
print(dup[dup > 1])


# =========================
# Aggregate replicates
# =========================
df_pheno = (
    df_pheno
    .groupby("Variety", as_index=False)   
    .mean(numeric_only=True)
)

df_pheno.to_csv(PHENO_PATH, index=False)

print("\nSaved cleaned phenotype:")
print(PHENO_PATH)


# =========================
# Tables
# =========================
GWAS_TABLE_NEW = CONFIG["data"]["gwas_table_newharvested"]
TAGLO_TABLE = CONFIG["paths"]["TAGLO_TABLE"]



Phenotype file: /Volumes/bmqg/default_bronze/fatemeh/final_project/csv_outputs_newharvested/matched_per_variety_STRICT_ONLY.csv
Raw phenotype shape: (95, 39)

Checking replicate varieties...
Total rows: 95
Unique varieties: 95

Replicates:
Series([], Name: count, dtype: int64)

Saved cleaned phenotype:
/Volumes/bmqg/default_bronze/fatemeh/final_project/csv_outputs_newharvested/matched_per_variety_STRICT_ONLY.csv


In [0]:
# =========================
# Output directories
# =========================
PNG_DIR = "/Volumes/bmqg/default_bronze/fatemeh/final_project/allelic_effect_plots_newharvested"
CSV_OUT = "/Volumes/bmqg/default_bronze/fatemeh/final_project/csv_outputs_newharvested/allelic_effect_csv_neharvested"

os.makedirs(PNG_DIR, exist_ok=True)
os.makedirs(CSV_OUT, exist_ok=True)


# =========================
# Traits
# =========================
traits = (
    spark.table(GWAS_TABLE_NEW)
    .select("trait")
    .distinct()
    .toPandas()["trait"]
    .tolist()
)

print("N traits:", len(traits))


# =========================
# Clusters
# =========================
clusters = effect_size_gemma.build_gwas_clusters(
    spark,
    GWAS_TABLE_NEW,
    p_thresh=1e-6,
    window_bp=500_000
)

print("Clusters shape:", clusters.shape)
display(clusters.head())


# =========================
# Prepare phenotype
# =========================
trait_set = set(traits)

df_pheno = df_pheno[
    ["Variety"] + [c for c in df_pheno.columns if c in trait_set]
]

print("Phenotype shape:", df_pheno.shape)


# =========================
# MAIN LOOP
# =========================
results = []

for r in clusters.itertuples():

    trait = r.trait
    chrom = r.chrom
    start_min = int(r.start_min)
    start_max = int(r.start_max)

    print(f"▶ {trait} | {chrom}:{start_min}-{start_max}")

    # -------------------------
    # skip if phenotype missing
    # -------------------------
    if trait not in df_pheno.columns:
        print("Skip: no phenotype")
        continue

    # -------------------------
    # taglos
    # -------------------------
    taglos = effect_size_gemma.get_cluster_taglos(
        spark,
        GWAS_TABLE_NEW,
        trait,
        chrom,
        start_min,
        start_max
    )

    if not taglos:
        print("Skip: no taglos")
        continue

    # -------------------------
    # dosage
    # -------------------------
    df = effect_size_gemma.build_dosage_df_from_taglo(
        spark,
        TAGLO_TABLE,
        taglos,
        df_pheno,
        trait
    )

    if df is None or df.empty:
        print("Skip: empty df")
        continue

    # -------------------------
    # linearity
    # -------------------------
    beh = effect_size_gemma.compute_dosage_linearity(df)

    if beh is None:
        print("Skip: no linearity result")
        continue

    # -------------------------
    # plot
    # -------------------------
    png, fc = effect_size_gemma.save_clean_allelic_png(
        df,
        trait,
        chrom,
        start_min,
        start_max,
        taglos,
        PNG_DIR
    )

    # -------------------------
    # collect
    # -------------------------
    results.append({
        "trait": trait,
        "chrom": chrom,
        "cluster_id": int(r.cluster_id),

        "start_min": start_min,
        "start_max": start_max,
        "n_snps": int(r.n_snps),

        "n_taglos": len(taglos),
        "taglo_ids": ",".join(map(str, taglos)),   

        "fold_change_4_vs_0": fc,
        "png": png,

        **beh
    })


# =========================
# Save
# =========================
results_df = pd.DataFrame(results)

out_csv = f"{CSV_OUT}/allelic_effect_summary_newharvested.csv"
results_df.to_csv(out_csv, index=False)

print("Saved:", out_csv)
print("Final shape:", results_df.shape)

display(results_df.head())

N traits: 38
Clusters shape: (589, 6)


trait,chrom,cluster_id,start_min,start_max,n_snps
(E)-2-Decenal,ST4.03ch07,1,44350000,44650000,12
"1-Hexanol, 2-ethyl-",ST4.03ch07,1,44350000,44650000,12
1-Octen-3-ol,ST4.03ch00,1,11250000,11250000,1
1-Octen-3-ol,ST4.03ch00,2,21200000,21250000,2
1-Octen-3-ol,ST4.03ch00,3,33700000,33700000,5


Phenotype shape: (95, 39)
▶ (E)-2-Decenal | ST4.03ch07:44350000-44650000
▶ 1-Hexanol, 2-ethyl- | ST4.03ch07:44350000-44650000
▶ 1-Octen-3-ol | ST4.03ch00:11250000-11250000
▶ 1-Octen-3-ol | ST4.03ch00:21200000-21250000
▶ 1-Octen-3-ol | ST4.03ch00:33700000-33700000
▶ 1-Octen-3-ol | ST4.03ch00:41250000-41250000
▶ 1-Octen-3-ol | ST4.03ch01:66900000-66900000
▶ 1-Octen-3-ol | ST4.03ch03:37850000-38100000
▶ 1-Octen-3-ol | ST4.03ch04:1700000-1750000
▶ 1-Octen-3-ol | ST4.03ch05:22500000-22500000
▶ 1-Octen-3-ol | ST4.03ch05:24200000-24200000
▶ 1-Octen-3-ol | ST4.03ch05:25600000-25600000
▶ 1-Octen-3-ol | ST4.03ch05:41100000-41100000
▶ 1-Octen-3-ol | ST4.03ch05:42050000-42050000
▶ 1-Octen-3-ol | ST4.03ch06:37550000-37550000
▶ 1-Octen-3-ol | ST4.03ch09:35800000-35800000
▶ 1-Octen-3-ol | ST4.03ch10:25050000-25050000
▶ 1-Octen-3-ol | ST4.03ch12:5150000-5150000
▶ 1-Octyn-3-ol | ST4.03ch07:44350000-44650000
▶ 1-Pentanol | ST4.03ch00:11350000-11350000
▶ 1-Pentanol | ST4.03ch00:30900000-30900000
▶ 1-Pent

/Volumes/bmqg/default_bronze/fatemeh/final_project/modules/effect_size_gemma.py:123: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


▶ 1-Pentanol | ST4.03ch04:68050000-68050000
▶ 1-Pentanol | ST4.03ch05:29300000-29300000
▶ 1-Pentanol | ST4.03ch06:19500000-19500000
▶ 1-Pentanol | ST4.03ch08:14100000-14100000
▶ 1-Pentanol | ST4.03ch08:26950000-26950000
▶ 1-Pentanol | ST4.03ch08:28500000-28500000
▶ 1-Pentanol | ST4.03ch09:54350000-55850000
▶ 1-Pentanol | ST4.03ch10:10200000-10200000
▶ 1-Pentanol | ST4.03ch10:13650000-13650000
▶ 1-Pentanol | ST4.03ch11:40950000-40950000
▶ 1-Penten-3-one | ST4.03ch00:41250000-41250000
▶ 1-Penten-3-one | ST4.03ch01:19850000-19850000
▶ 1-Penten-3-one | ST4.03ch01:73900000-74150000
▶ 1-Penten-3-one | ST4.03ch02:25100000-25500000
▶ 1-Penten-3-one | ST4.03ch02:46650000-46750000
▶ 1-Penten-3-one | ST4.03ch06:46300000-46300000
▶ 1-Penten-3-one | ST4.03ch08:2550000-2550000
▶ 1-Penten-3-one | ST4.03ch10:50750000-50850000


...

/Volumes/bmqg/default_bronze/fatemeh/final_project/modules/effect_size_gemma.py:123: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


▶ Linalool | ST4.03ch02:24400000-24400000
▶ Linalool | ST4.03ch02:48300000-48300000
▶ Linalool | ST4.03ch06:52700000-52750000
▶ Linalool | ST4.03ch09:56000000-56000000
▶ Linalool | ST4.03ch10:59400000-59700000
▶ Linalool | ST4.03ch11:2600000-2600000
▶ Linalool | ST4.03ch11:6550000-6650000
▶ Linalool | ST4.03ch11:21050000-21300000
▶ Linalool | ST4.03ch12:47450000-47450000
▶ Nonanal | ST4.03ch01:350000-400000
▶ Nonanal | ST4.03ch01:10800000-11250000
▶ Nonanal | ST4.03ch01:45200000-45200000
▶ Nonanal | ST4.03ch01:53800000-54000000
▶ Nonanal | ST4.03ch01:57200000-57200000
▶ Nonanal | ST4.03ch01:87400000-87400000
▶ Nonanal | ST4.03ch02:16000000-16000000
▶ Nonanal | ST4.03ch03:37850000-38100000
▶ Nonanal | ST4.03ch06:1900000-1900000
▶ Nonanal | ST4.03ch11:8850000-8850000
▶ Nonanal | ST4.03ch11:42600000-43100000
▶ Nonanal | ST4.03ch12:5150000-5150000
▶ Nonanal | ST4.03ch12:13800000-13800000
▶ Nonanal | ST4.03ch12:15150000-15150000
▶ Nonanal | ST4.03ch12:29800000-29800000
▶ Nonanal | ST4.03ch1

/Volumes/bmqg/default_bronze/fatemeh/final_project/modules/effect_size_gemma.py:123: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


▶ Tetradecane | ST4.03ch01:28850000-28850000
▶ Tetradecane | ST4.03ch01:73300000-73300000
▶ Tetradecane | ST4.03ch02:45700000-45750000
▶ Tetradecane | ST4.03ch03:37900000-38100000
▶ Tetradecane | ST4.03ch06:750000-750000
▶ Tetradecane | ST4.03ch08:38950000-38950000
▶ Tetradecane | ST4.03ch10:25050000-25050000
▶ Tetradecane | ST4.03ch11:42600000-42600000
▶ Tetradecane | ST4.03ch12:49300000-49300000
▶ nonanoic acid | ST4.03ch01:81750000-81750000
▶ nonanoic acid | ST4.03ch02:26000000-26000000
Saved: /Volumes/bmqg/default_bronze/fatemeh/final_project/csv_outputs_newharvested/allelic_effect_csv_neharvested/allelic_effect_summary_newharvested.csv
Final shape: (552, 20)


trait,chrom,cluster_id,start_min,start_max,n_snps,n_taglos,taglo_ids,fold_change_4_vs_0,png,status,n_groups,dosages_used,spearman_rho,spearman_p,linear_slope,linear_r2,linear_p,delta_extreme,dosage_shape
(E)-2-Decenal,ST4.03ch07,1,44350000,44650000,12,11,"336975,336993,336982,337231,337226,337228,336974,336979,337230,337225,337227",null,/Volumes/bmqg/default_bronze/fatemeh/final_project/allelic_effect_plots_newharvested/(E)-2-Decenal_ST4.03ch07_44350000_44650000.png,ok,3,"List(2, 3, 4)",-0.5,0.6666666666666667,-872966.125,0.6129863964417626,0.4274439529540519,-1745932.25,non_monotonic
"1-Hexanol, 2-ethyl-",ST4.03ch07,1,44350000,44650000,12,11,"336975,336993,337227,336982,337231,337228,336974,336979,337230,337226,337225",null,"/Volumes/bmqg/default_bronze/fatemeh/final_project/allelic_effect_plots_newharvested/1-Hexanol,_2-ethyl-_ST4.03ch07_44350000_44650000.png",ok,3,"List(2, 3, 4)",-0.5,0.6666666666666667,-872966.125,0.6129863964417626,0.4274439529540519,-1745932.25,non_monotonic
1-Octen-3-ol,ST4.03ch00,1,11250000,11250000,1,4,"6476,6478,6475,6477",null,/Volumes/bmqg/default_bronze/fatemeh/final_project/allelic_effect_plots_newharvested/1-Octen-3-ol_ST4.03ch00_11250000_11250000.png,ok,2,"List(0, 1)",0.9999999999999999,null,28880.0,1.0,0.0,28880.0,monotonic
1-Octen-3-ol,ST4.03ch00,2,21200000,21250000,2,2,"11481,11522",null,/Volumes/bmqg/default_bronze/fatemeh/final_project/allelic_effect_plots_newharvested/1-Octen-3-ol_ST4.03ch00_21200000_21250000.png,ok,2,"List(0, 2)",0.9999999999999999,null,22674.25,1.0,0.0,45348.5,monotonic
1-Octen-3-ol,ST4.03ch00,3,33700000,33700000,5,4,"16446,16443,16444,16441",null,/Volumes/bmqg/default_bronze/fatemeh/final_project/allelic_effect_plots_newharvested/1-Octen-3-ol_ST4.03ch00_33700000_33700000.png,ok,4,"List(0, 1, 2, 3)",1.0,0.0,25202.95,0.790874147920517,0.11068894760015655,77973.5,monotonic
